# 02 – Detection Training with YOLOv8 (1-class: trash)

This notebook walks through labelling, dataset preparation, training, and
evaluation of the first stage – a single-class **trash detector** built on
YOLOv8.

## 1 · Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("__file__").resolve().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from trash_detection.yolo import (
    write_dataset_yaml,
    annotation_to_yolo,
    read_yolo_label,
    yolo_to_bbox,
)
from trash_detection.viz import draw_yolo_annotations
import cv2

DATASET_ROOT = REPO_ROOT / "datasets" / "detection"
YAML_PATH    = REPO_ROOT / "trash_yolo.yaml"

print("Repo root   :", REPO_ROOT)
print("Dataset root:", DATASET_ROOT)

## 2 · Labelling Workflow

Labels are created with **LabelImg** (or exported from **Roboflow**).

### LabelImg quick-start

```bash
pip install labelImg
labelImg datasets/detection/images/train  # open image directory
```

Choose **YOLO** as the save format.  Set the label to `trash` (class 0).

### Expected label format (one row per bounding box)

```
<class_id> <cx_norm> <cy_norm> <w_norm> <h_norm>
```

Example – a box occupying the centre 20 % of the image:

```
0 0.500000 0.500000 0.200000 0.200000
```

All values are normalised to [0, 1] relative to the image dimensions.

In [ ]:
# Demonstrate annotation_to_yolo helper
example_line = annotation_to_yolo(
    x1=200, y1=150, x2=400, y2=350,
    img_w=640, img_h=480,
    class_id=0,
)
print("YOLO annotation line:", example_line)

# Round-trip: parse it back
import tempfile, os
with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as tf:
    tf.write(example_line)
    tmp_path = tf.name

parsed = read_yolo_label(tmp_path)
print("Parsed back        :", parsed)
os.unlink(tmp_path)

## 3 · Dataset Structure for YOLO

YOLOv8 expects the following layout:

```
datasets/detection/
├── images/
│   ├── train/   ← training images
│   ├── val/     ← validation images
│   └── test/    ← test images
└── labels/
    ├── train/   ← .txt label files (same stem as image)
    ├── val/
    └── test/
```

Use `scripts/merge_yolo_datasets.py` if you are combining multiple sources.

In [ ]:
# Write (or overwrite) the dataset YAML
write_dataset_yaml(
    path=YAML_PATH,
    dataset_root=DATASET_ROOT,
    nc=1,
    names=["trash"],
)
print(YAML_PATH.read_text())

## 4 · Training

Run the training command from the repository root.  Adjust `epochs`, `imgsz`,
and `batch` to your hardware.

```bash
yolo detect train \
    model=yolov8n.pt \
    data=trash_yolo.yaml \
    epochs=100 \
    imgsz=640 \
    batch=16 \
    name=trash_detector_v1 \
    project=runs/detect
```

Or via the Python API:

In [ ]:
# Uncomment to launch training directly from the notebook

# from ultralytics import YOLO
#
# model = YOLO("yolov8n.pt")
# results = model.train(
#     data=str(YAML_PATH),
#     epochs=100,
#     imgsz=640,
#     batch=16,
#     name="trash_detector_v1",
#     project="runs/detect",
# )
print("Training command ready – uncomment the cells above to start.")

## 5 · Evaluation

After training completes, evaluate on the test split:

```bash
yolo detect val \
    model=runs/detect/trash_detector_v1/weights/best.pt \
    data=trash_yolo.yaml \
    split=test
```

Key metrics to inspect in `runs/detect/trash_detector_v1/`:

| File | Metric |
|------|--------|
| `results.csv` | mAP@50, mAP@50-95, Precision, Recall per epoch |
| `PR_curve.png` | Precision-Recall curve |
| `confusion_matrix.png` | Confusion matrix |
| `val_batch*.jpg` | Sample predictions on validation images |

In [ ]:
# Uncomment to run evaluation from the notebook

# from ultralytics import YOLO
#
# BEST_WEIGHTS = REPO_ROOT / "runs" / "detect" / "trash_detector_v1" / "weights" / "best.pt"
# model = YOLO(str(BEST_WEIGHTS))
# metrics = model.val(data=str(YAML_PATH), split="test")
# print(f"mAP@50     : {metrics.box.map50:.4f}")
# print(f"mAP@50-95  : {metrics.box.map:.4f}")
print("Evaluation command ready – uncomment cells above after training.")

## 6 · Notes on Hyperparameter Tuning

### Model size trade-offs

| Model | Params | mAP (COCO) | Latency |
|-------|--------|------------|---------|
| yolov8n.pt | 3.2 M | 37.3 | fastest |
| yolov8s.pt | 11.2 M | 44.9 | fast |
| yolov8m.pt | 25.9 M | 50.2 | moderate |

Start with `yolov8n` for rapid iteration, promote to `yolov8s` or `yolov8m`
when dataset is finalised.

### Key hyperparameters

```yaml
lr0: 0.01       # initial learning rate
lrf: 0.01       # final lr = lr0 * lrf
momentum: 0.937
weight_decay: 0.0005
warmup_epochs: 3
augment: True   # built-in mosaic + flip augmentation
```

Use `yolo detect train ... cfg=custom.yaml` to override defaults.

### Early stopping

Pass `patience=20` to stop training if mAP@50 does not improve for 20 epochs.